In [ ]:
# ==================== 각자 수정 ====================
from pathlib import Path

VMMR_ROOT  = Path(r"/Users/yuhuiseon/Downloads/dataset/VMMRdb")   # 내 컴퓨터의 VMMRdb 폴더
CACHE_ROOT = Path("cache")       # 02 노트북에서 만든 리사이즈 캐시 (없으면 원본을 읽음)
RUNS_DIR   = Path("runs")        # 실험 결과 저장 위치
SMOKE_TEST = False               # True: 아주 작은 데이터로 코드가 도는지만 확인 (결과는 runs_smoke/에 저장)
# ==================================================

# 03. 학습 · 평가 — 불균형 해소와 계층형 분류

02 노트북이 만든 `splits/`를 읽어서 모델을 학습하고 평가한다. 모든 실험은 같은 분할, 같은 백본, 같은 평가 함수를 쓴다. **바꾸는 것은 설정(CFG) 한두 개뿐**이다.

| 파트 | 내용 |
|---|---|
| 1 | 설정 · 장치 · 데이터 |
| 2 | 모델: ImageNet 사전학습 ResNet18 + Flat 머리 / 계층(hard routing) 머리 |
| 3 | **불균형 툴킷**: 손실 가중 · 손실 형태 · 샘플링 · DRW · 사후 보정 · cRT |
| 4 | 학습 루프 (val Macro-F1 기준 최적 epoch 선택) |
| 5 | 평가 함수 (Macro-F1, head/medium/tail recall, 형제차 혼동, 계층 오류 분해) |
| 6 | 실험 실행: 불균형 비교 → 혼동 기반 그룹 생성 → 계층 비교 |
| 7 | 결과 비교 표와 그림 |

> **백본에 대해:** 모든 모델이 ImageNet 사전학습 ResNet18을 똑같이 쓴다. 그래서 비교의 공정성에는 영향이 없다. 다만 보고서에는 "백본은 사전학습 ResNet18"이라고 적어야 한다.

## 1. 설정 · 장치 · 데이터

In [ ]:
import copy, json, math, os, random, time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image
from sklearn.cluster import SpectralClustering
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score
from tqdm.auto import tqdm

for f in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if any(f == x.name for x in mpl.font_manager.fontManager.ttflist):
        mpl.rcParams["font.family"] = f; break
mpl.rcParams["axes.unicode_minus"] = False

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
if SMOKE_TEST:
    RUNS_DIR = Path("runs_smoke")
RUNS_DIR.mkdir(exist_ok=True)
print("Device:", DEVICE, "| 결과 폴더:", RUNS_DIR)

### 기본 설정 `BASE`

모든 실험은 `BASE`에서 시작하고, 실험마다 바꿀 값만 덮어쓴다(6장).

| 키 | 의미 |
|---|---|
| `img_size`, `batch_size`, `epochs`, `lr`, `weight_decay`, `patience` | 일반 학습 설정. `patience`는 val Macro-F1이 몇 epoch 동안 안 오르면 멈출지 |
| `train_fraction`, `eval_fraction` | 데이터 일부만 쓰기 (빠른 실험·점검용) |
| `num_workers` | 데이터 로딩 프로세스 수. **Mac·Windows의 Jupyter에서는 0**이어야 한다. Linux+CUDA면 4~8 |
| `hier` | `none`(Flat) · `make` · `body` · `confusion` |
| 나머지 | 불균형 툴킷 손잡이 (3장 표 참고) |

In [ ]:
BASE = dict(
    img_size=224, batch_size=64, epochs=12, lr=3e-4, weight_decay=1e-4, patience=4,
    train_fraction=1.0, eval_fraction=1.0, num_workers=0,
    hier="none", n_conf_groups=6,
    # ---- 불균형 툴킷 ----
    weighting="none", alpha=1.0, beta=0.999, manual_w=None,   # 손실 가중
    loss="ce", gamma=2.0, tau=1.0,                            # 손실 형태
    q=1.0,                                                    # 샘플링 (1=그대로, 0.5=제곱근, 0=클래스 균형)
    drw_epoch=None,                                           # Deferred Re-Weighting 시작 epoch
    posthoc_tau=0.0,                                          # 추론 시 logit 보정
    crt_epochs=0, from_run=None,                              # cRT / 다른 실험의 체크포인트 재사용
)
SEEDS = [0, 1, 2]
if SMOKE_TEST:
    BASE.update(img_size=112, batch_size=32, epochs=1, train_fraction=0.01, eval_fraction=0.05, patience=1)
    SEEDS = [0]

In [ ]:
SPLIT_DIR = Path("splits")
CLASS_INFO = pd.read_csv(SPLIT_DIR / "class_info.csv").sort_values("model_idx").reset_index(drop=True)
SPLITS = {k: pd.read_csv(SPLIT_DIR / f"{k}.csv") for k in ["train", "val", "test"]}
N_CLASSES = len(CLASS_INFO)
CLASS_NAMES = CLASS_INFO.make_model.tolist()
TRAIN_COUNTS = np.bincount(SPLITS["train"].model_idx, minlength=N_CLASSES)   # 가중치·prior는 항상 train 개수로만 계산

CACHE_DIR = CACHE_ROOT / "vmmr_256"
USE_CACHE = CACHE_DIR.is_dir() and (CACHE_DIR / SPLITS["train"].rel_path.iloc[0]).is_file()
IMG_ROOT = CACHE_DIR if USE_CACHE else VMMR_ROOT
assert (IMG_ROOT / SPLITS["train"].rel_path.iloc[0]).is_file(), f"이미지를 찾을 수 없습니다: {IMG_ROOT}"
print(f"이미지 위치: {IMG_ROOT} ({'캐시' if USE_CACHE else '원본'})")
print({k: len(v) for k, v in SPLITS.items()}, f"| train 최대/최소 {TRAIN_COUNTS.max()}/{TRAIN_COUNTS.min()}")

TWIN_PAIRS = [("chevrolet/silverado", "gmc/sierra_1500"), ("chevrolet/tahoe", "gmc/yukon_1500"), ("bmw/325i", "bmw/328i")]

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)     # ImageNet 값 (사전학습 백본과 맞춤)

def make_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    eval_tf = transforms.Compose([                              # 평가에는 무작위 변환을 쓰지 않는다
        transforms.Resize(round(img_size * 256 / 224)), transforms.CenterCrop(img_size),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return train_tf, eval_tf

class CarDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = [IMG_ROOT / p for p in df.rel_path]
        self.y = df.model_idx.to_numpy()
        self.transform = transform
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.transform(im.convert("RGB"))
        return x, int(self.y[i])

def subsample(df, frac, seed):
    """차종 비율을 유지하며 일부만 뽑는다 (차종마다 최소 1장)."""
    if frac >= 1:
        return df
    return (df.groupby("model_idx", group_keys=False)
              .apply(lambda g: g.sample(max(1, round(len(g) * frac)), random_state=seed)))

def make_loaders(cfg, seed):
    train_tf, eval_tf = make_transforms(cfg.img_size)
    tr = subsample(SPLITS["train"], cfg.train_fraction, seed)
    va = subsample(SPLITS["val"], cfg.eval_fraction, seed)
    te = subsample(SPLITS["test"], cfg.eval_fraction, seed)
    kw = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=DEVICE.type == "cuda")
    g = torch.Generator().manual_seed(seed)
    train_ds = CarDataset(tr, train_tf)
    if cfg.q < 1:          # 반복 샘플링: 클래스 c의 이미지가 뽑힐 확률 ∝ n_c^(q-1) → 클래스 전체 확률 ∝ n_c^q
        w = TRAIN_COUNTS[train_ds.y].astype(float) ** (cfg.q - 1)
        sampler = WeightedRandomSampler(torch.as_tensor(w), num_samples=len(w), replacement=True, generator=g)
        train_loader = DataLoader(train_ds, sampler=sampler, **kw)
    else:
        train_loader = DataLoader(train_ds, shuffle=True, generator=g, **kw)
    return {"train": train_loader,
            "train_eval": DataLoader(CarDataset(tr, eval_tf), shuffle=False, **kw),   # cRT 특징 추출용
            "val": DataLoader(CarDataset(va, eval_tf), shuffle=False, **kw),
            "test": DataLoader(CarDataset(te, eval_tf), shuffle=False, **kw)}

## 2. 모델 — 공통 백본 + 두 가지 머리

- **백본:** ImageNet 사전학습 ResNet18. 마지막 층을 떼면 이미지 1장 → 512차원 특징이 나온다.
- **Flat 머리:** 512 → 50. 50개 차종을 한 번에 고른다. (H0)
- **계층 머리 (hard routing):**
  - **그룹 머리:** 512 → K. 1단계에서 그룹을 고른다.
  - **그룹별 차종 머리:** 그룹마다 512 → (그룹 안 차종 수). 2단계에서 **1단계가 고른 그룹의 머리만** 써서 차종을 고른다.
  - 그룹을 무엇으로 정할지는 "차종 → 그룹" 대응표 하나로 바꾼다: `make`(10) · `body`(6) · `confusion`(K, 6장에서 생성).

In [ ]:
class CarNet(nn.Module):
    def __init__(self, n_classes, groups=None):
        super().__init__()
        bb = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        feat_dim = bb.fc.in_features
        bb.fc = nn.Identity()
        self.backbone = bb
        self.groups = groups
        if groups is None:
            self.head = nn.Linear(feat_dim, n_classes)
        else:
            self.group_head = nn.Linear(feat_dim, len(groups))
            self.sub_heads = nn.ModuleList([nn.Linear(feat_dim, len(g)) for g in groups])

    def heads(self, f):
        if self.groups is None:
            return {"logits": self.head(f)}
        return {"group_logits": self.group_head(f), "sub_logits": [h(f) for h in self.sub_heads]}

    def forward(self, x):
        return self.heads(self.backbone(x))

    def reset_heads(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                m.reset_parameters()

def load_groups(hier):
    """차종 → 그룹 대응을 그룹별 차종 목록(list of list)으로 돌려준다. none이면 None."""
    if hier == "none":
        return None
    if hier in ("make", "body"):
        col = f"{hier}_idx"
        return [CLASS_INFO.model_idx[CLASS_INFO[col] == k].tolist() for k in sorted(CLASS_INFO[col].unique())]
    if hier == "confusion":
        f = RUNS_DIR / "confusion_groups.json"
        assert f.exists(), "혼동 기반 그룹이 없습니다. 6-2 셀(기준 모델 학습 → 그룹 생성)을 먼저 실행하세요."
        return json.loads(f.read_text())["groups"]
    raise ValueError(hier)

## 3. 불균형 툴킷 — 이 프로젝트의 핵심

train 기준으로 가장 많은 차종과 가장 적은 차종이 약 26배 차이 난다. 불균형을 다루는 방법을 **층위별로** 모아, 모두 설정값 하나로 켜고 끌 수 있게 했다.

| 층위 | 기법 | 설정 | 하는 일 | 출처 |
|---|---|---|---|---|
| 손실 가중 | 역빈도 | `weighting="inv"`, `alpha` | 차종 c의 손실에 $(1/n_c)^{\alpha}$를 곱한다. α=1이 수업 코드와 같고, α=0이면 가중치 없음 | 수업 코드 |
| 손실 가중 | Class-Balanced | `weighting="cb"`, `beta` | $w_c \propto \frac{1-\beta}{1-\beta^{n_c}}$. 사진이 많아질수록 새 사진의 가치가 줄어드는 점을 반영한다 | Cui et al., 2019 |
| 손실 가중 | 수동 보정 | `manual_w={"gmc/sierra_1500": 2.0}` | 특정 차종의 가중치를 직접 올리거나 내린다 | — |
| 손실 형태 | Focal | `loss="focal"`, `gamma` | 이미 잘 맞히는 쉬운 사진의 손실을 $(1-p)^{\gamma}$로 줄여 어려운 사진에 집중한다 | Lin et al., 2017 |
| 손실 형태 | Logit-adjusted / Balanced Softmax | `loss="la"`, `tau` | 학습할 때 logit에 $\tau \log(\text{prior})$를 더한다. 큰 차종이 "공짜로" 이기는 몫을 미리 빼 주는 효과다 (τ=1이면 Balanced Softmax) | Menon et al., 2021; Ren et al., 2020 |
| 샘플링 | 반복 샘플링 | `q` | 차종 c가 배치에 뽑힐 확률을 $n_c^{q}$에 비례하게 한다. q=1 그대로, **q=0.5 제곱근**, q=0 클래스 균형 | Kang et al., 2020 |
| 스케줄 | DRW | `drw_epoch` | 처음에는 가중치 없이 특징을 배우고, `drw_epoch`부터 가중치를 켠다 | Cao et al., 2019 |
| 사후 보정 | Post-hoc logit adjustment | `posthoc_tau` | **다시 학습하지 않고** 예측할 때만 logit에서 $\tau \log(\text{prior})$를 뺀다 | Menon et al., 2021 |
| 분리 학습 | cRT | `crt_epochs`, `from_run` | 학습된 백본은 고정하고, 마지막 분류층만 **클래스 균형 샘플링**으로 다시 학습한다 | Kang et al., 2020 |

- 가중치와 prior는 모두 **train 분할의 개수**로만 계산한다. val과 test 정보가 새지 않게 하기 위해서다.
- 가중치는 차종 평균이 1이 되도록 맞춘다.
- 계층 모델은 그룹 손실과 그룹 안 차종 손실에 같은 방식을 각각 적용한다. 그룹의 개수는 그룹에 속한 차종 개수의 합이다.

In [ ]:
def class_weights(counts, cfg, names=None):
    n = torch.as_tensor(counts, dtype=torch.float32)
    if cfg.weighting == "none":
        w = torch.ones_like(n)
    elif cfg.weighting == "inv":
        w = (1.0 / n) ** cfg.alpha
    elif cfg.weighting == "cb":
        w = (1 - cfg.beta) / (1 - cfg.beta ** n)
    else:
        raise ValueError(cfg.weighting)
    w = w / w.mean()
    if cfg.manual_w and names is not None:
        for name, mult in cfg.manual_w.items():
            if name in names:
                w[names.index(name)] *= mult
    return w

def imb_loss(logits, y, w, log_prior, cfg):
    """손실 형태(ce / focal / la)와 클래스 가중치를 한 번에 적용한다."""
    if cfg.loss == "la":
        logits = logits + cfg.tau * log_prior
    logp = F.log_softmax(logits, dim=1)
    logpt = logp.gather(1, y[:, None]).squeeze(1)
    loss = -logpt
    if cfg.loss == "focal":
        loss = loss * (1 - logpt.exp()).pow(cfg.gamma)
    if w is not None:
        loss = loss * w[y]
    return loss.mean()

class Context:
    """실험 하나에 필요한 가중치·prior·그룹 정보를 장치(device) 위에 모아 둔다."""
    def __init__(self, cfg, groups):
        dev = DEVICE
        self.groups = groups
        self.cls_w = class_weights(TRAIN_COUNTS, cfg, CLASS_NAMES).to(dev)
        self.log_prior = torch.log(torch.as_tensor(TRAIN_COUNTS / TRAIN_COUNTS.sum(), dtype=torch.float32)).to(dev)
        if groups is None:
            return
        cls2grp, local = np.zeros(N_CLASSES, int), np.zeros(N_CLASSES, int)
        for k, g in enumerate(groups):
            for j, c in enumerate(g):
                cls2grp[c], local[c] = k, j
        self.cls2grp = torch.as_tensor(cls2grp, device=dev)
        self.local = torch.as_tensor(local, device=dev)
        self.group_members = [torch.as_tensor(g, device=dev) for g in groups]
        gcounts = np.array([TRAIN_COUNTS[g].sum() for g in groups])
        self.grp_w = class_weights(gcounts, cfg).to(dev)
        self.grp_log_prior = torch.log(torch.as_tensor(gcounts / gcounts.sum(), dtype=torch.float32)).to(dev)
        self.sub_w = [class_weights(TRAIN_COUNTS[g], cfg).to(dev) for g in groups]
        self.sub_log_prior = [torch.log(torch.as_tensor(TRAIN_COUNTS[g] / TRAIN_COUNTS[g].sum(), dtype=torch.float32)).to(dev)
                              for g in groups]

def compute_loss(out, y, ctx, cfg, weighted=True):
    if ctx.groups is None:
        return imb_loss(out["logits"], y, ctx.cls_w if weighted else None, ctx.log_prior, cfg)
    g = ctx.cls2grp[y]
    loss = imb_loss(out["group_logits"], g, ctx.grp_w if weighted else None, ctx.grp_log_prior, cfg)
    sub = 0.0
    for k in range(len(ctx.groups)):
        m = g == k
        if m.any():      # 2단계는 "정답 그룹"의 머리로 학습한다
            sub = sub + imb_loss(out["sub_logits"][k][m], ctx.local[y[m]],
                                 ctx.sub_w[k] if weighted else None, ctx.sub_log_prior[k], cfg) * m.sum()
    return loss + sub / len(y)

## 4. 예측과 평가 함수 (모든 모델 공통)

**예측 방식**
- **Flat:** 50개 점수 중 가장 큰 것을 고른다.
- **계층 (hard routing):** 1단계에서 그룹 하나를 고르고, 그 그룹의 머리 안에서만 차종을 고른다.
  - Top-3와 "soft 결합 Top-1"은 참고용으로 $P(\text{그룹}) \times P(\text{차종}\mid\text{그룹})$ 점수로 계산한다.
  - 이렇게 하면 hard routing과 soft 결합을 같은 모델에서 비교할 수 있다.

**지표**

| 지표 | 의미 |
|---|---|
| `top1`, `top3` | 전체 정확도 |
| **`macro_f1`** (주 지표) | 50개 차종을 따로 채점한 F1의 평균. 차종마다 똑같이 1표 |
| `bal_acc` | 차종별 recall의 평균 |
| `recall_head / medium / tail` | 이미지 수 구간별 평균 recall. **불균형 해소 효과가 가장 직접적으로 보이는 지표** |
| `make_acc`, `body_acc` | 예측한 차종의 제조사·차체 유형이 맞은 비율. "덜 나쁜 실수"를 보는 지표 |
| `twin_confusion` | 형제차·트림 쌍끼리 서로 착각한 비율 |
| `stage1_acc`, `stage2_acc_given1`, `err_from_stage1` | 계층 전용: 1단계 정확도, 1단계가 맞았을 때 2단계 정확도, 전체 오답 중 1단계에서 생긴 비율 |

In [ ]:
@torch.no_grad()
def predict(model, loader, ctx, cfg):
    model.eval()
    ys, preds, joints, gs, ghats = [], [], [], [], []
    pt = cfg.posthoc_tau
    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        if ctx.groups is None:
            logp = F.log_softmax(out["logits"] - pt * ctx.log_prior, dim=1)
            pred, joint = logp.argmax(1), logp
        else:
            glp = F.log_softmax(out["group_logits"] - pt * ctx.grp_log_prior, dim=1)
            ghat = glp.argmax(1)
            joint = torch.full((len(x), N_CLASSES), -1e9, device=DEVICE)
            pred = torch.zeros(len(x), dtype=torch.long, device=DEVICE)
            for k, members in enumerate(ctx.group_members):
                slp = F.log_softmax(out["sub_logits"][k] - pt * ctx.sub_log_prior[k], dim=1)
                joint[:, members] = glp[:, k:k + 1] + slp
                m = ghat == k
                if m.any():
                    pred[m] = members[slp[m].argmax(1)]
            gs.append(ctx.cls2grp[y.to(DEVICE)].cpu()); ghats.append(ghat.cpu())
        ys.append(y); preds.append(pred.cpu()); joints.append(joint.float().cpu())
    res = {"y": torch.cat(ys).numpy(), "pred": torch.cat(preds).numpy(), "joint": torch.cat(joints).numpy()}
    if gs:
        res["g"], res["ghat"] = torch.cat(gs).numpy(), torch.cat(ghats).numpy()
    return res

def compute_metrics(r):
    y, pred, joint = r["y"], r["pred"], r["joint"]
    labels = np.arange(N_CLASSES)
    top3 = np.argsort(-joint, axis=1)[:, :3]
    rec = np.array([(pred[y == c] == c).mean() if (y == c).any() else np.nan for c in labels])
    m = {"n": int(len(y)), "top1": float((pred == y).mean()), "top3": float((top3 == y[:, None]).any(1).mean()),
         "macro_f1": float(f1_score(y, pred, labels=labels, average="macro", zero_division=0)),
         "bal_acc": float(np.nanmean(rec))}
    for grp in ["head", "medium", "tail"]:
        m[f"recall_{grp}"] = float(np.nanmean(rec[(CLASS_INFO.freq_group == grp).to_numpy()]))
    mk, bd = CLASS_INFO.make_idx.to_numpy(), CLASS_INFO.body_idx.to_numpy()
    m["make_acc"] = float((mk[pred] == mk[y]).mean())
    m["body_acc"] = float((bd[pred] == bd[y]).mean())
    twin = {}
    for a, b in TWIN_PAIRS:
        ia, ib = CLASS_NAMES.index(a), CLASS_NAMES.index(b)
        sel = (y == ia) | (y == ib)
        twin[f"{a}<->{b}"] = float(((y == ia) & (pred == ib) | (y == ib) & (pred == ia))[sel].mean()) if sel.any() else np.nan
    m["twin_confusion"] = twin
    if "g" in r:
        ok1 = r["g"] == r["ghat"]
        wrong = pred != y
        m["stage1_acc"] = float(ok1.mean())
        m["stage2_acc_given1"] = float((pred == y)[ok1].mean()) if ok1.any() else np.nan
        m["err_from_stage1"] = float((~ok1 & wrong).sum() / max(1, wrong.sum()))
        m["soft_top1"] = float((joint.argmax(1) == y).mean())
    m["per_class_recall"] = rec.tolist()
    m["per_class_f1"] = f1_score(y, pred, labels=labels, average=None, zero_division=0).tolist()
    return m

## 5. 학습 루프

- **최적화:** AdamW + cosine 학습률 감소
- **모델 선택:** 매 epoch 끝에 **val Macro-F1**을 재고, 가장 좋았던 epoch의 가중치를 남긴다. `patience` epoch 동안 오르지 않으면 멈춘다.
- **DRW를 쓸 때:** 가중치가 켜진 뒤의 epoch에서만 최적 epoch를 고른다.
- **cRT:** 학습이 끝난 백본을 고정한다. train 이미지의 특징을 **한 번만** 뽑고, 그 위에서 분류층만 클래스 균형 샘플링으로 다시 학습한다. 그래서 빠르다.
- **`from_run`:** 다른 실험의 체크포인트를 불러와 학습을 건너뛴다. 사후 보정과 cRT를 기준 모델 위에 얹을 때 쓴다.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def fit(model, loaders, ctx, cfg):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs * len(loaders["train"]))
    best_f1, best_state, bad, curves = -1.0, None, 0, []
    for ep in range(cfg.epochs):
        weighted = cfg.drw_epoch is None or ep >= cfg.drw_epoch
        model.train(); total, n = 0.0, 0
        for x, y in tqdm(loaders["train"], desc=f"epoch {ep + 1}/{cfg.epochs}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = compute_loss(model(x), y, ctx, cfg, weighted)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item() * len(y); n += len(y)
        vm = compute_metrics(predict(model, loaders["val"], ctx, cfg))
        curves.append({"epoch": ep + 1, "train_loss": total / n, "val_top1": vm["top1"], "val_macro_f1": vm["macro_f1"]})
        print(f"  epoch {ep + 1}: loss {total / n:.4f} | val top1 {vm['top1']:.3f} | val macro-F1 {vm['macro_f1']:.3f}")
        if not weighted:
            continue
        if vm["macro_f1"] > best_f1:
            best_f1, bad = vm["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience:
                print("  조기 종료"); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return pd.DataFrame(curves)

@torch.no_grad()
def extract_features(model, loader):
    model.eval(); fs, ys = [], []
    for x, y in tqdm(loader, desc="특징 추출", leave=False):
        fs.append(model.backbone(x.to(DEVICE)).cpu()); ys.append(y)
    return torch.cat(fs), torch.cat(ys)

def classifier_retrain(model, loaders, ctx, cfg, seed):
    """cRT: 백본 고정, 분류층만 클래스 균형 샘플링 + 일반 CE로 다시 학습."""
    feats, ys = extract_features(model, loaders["train_eval"])
    model.reset_heads()
    head_params = [p for n_, p in model.named_parameters() if not n_.startswith("backbone.")]
    opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=cfg.weight_decay)
    plain = SimpleNamespace(**{**vars(cfg), "loss": "ce"})
    w = TRAIN_COUNTS[ys.numpy()].astype(float) ** -1.0
    g = torch.Generator().manual_seed(seed)
    best_f1, best_state = -1.0, None
    for ep in range(cfg.crt_epochs):
        idx = torch.multinomial(torch.as_tensor(w), len(w), replacement=True, generator=g)
        model.train()
        for b in range(0, len(idx), 256):
            j = idx[b:b + 256]
            out = model.heads(feats[j].to(DEVICE))
            loss = compute_loss(out, ys[j].to(DEVICE), ctx, plain, weighted=False)
            opt.zero_grad(); loss.backward(); opt.step()
        vm = compute_metrics(predict(model, loaders["val"], ctx, cfg))
        if vm["macro_f1"] > best_f1:
            best_f1 = vm["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"  cRT 완료: val macro-F1 {best_f1:.3f}")
    model.load_state_dict(best_state)

## 6. 실험 실행

`run(name, overrides, seed)` 하나로 모든 실험을 돌린다.
- 결과는 `RUNS_DIR/<실험이름>/seed<k>/`에 저장된다: `config.json`, `metrics.json`(test), `val_metrics.json`, `val_confusion.npy`, `test_confusion.npy`, `test_preds.csv`, `curves.csv`, `best.pt`
- **이미 `metrics.json`이 있는 실험은 건너뛴다.** 그래서 중간에 멈춰도 다시 실행하면 이어서 돈다.
- 새 실험을 추가하려면 아래 사전에 `"이름": dict(바꿀 설정)` 한 줄만 넣으면 된다.

In [ ]:
def run(name, overrides, seed):
    cfg = SimpleNamespace(**{**BASE, **overrides})
    out_dir = RUNS_DIR / name / f"seed{seed}"
    if (out_dir / "metrics.json").exists():
        print(f"[건너뜀] {name} seed{seed}")
        return json.loads((out_dir / "metrics.json").read_text())
    print(f"=== {name} seed{seed} | {overrides}")
    out_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    t0 = time.time()
    groups = load_groups(cfg.hier)
    ctx = Context(cfg, groups)
    loaders = make_loaders(cfg, seed)
    model = CarNet(N_CLASSES, groups).to(DEVICE)
    curves = pd.DataFrame()
    if cfg.from_run:
        src = RUNS_DIR / cfg.from_run / f"seed{seed}" / "best.pt"
        assert src.exists(), f"{src}가 없습니다. '{cfg.from_run}' 실험을 먼저 돌리세요."
        model.load_state_dict(torch.load(src, map_location=DEVICE))
    else:
        curves = fit(model, loaders, ctx, cfg)
    if cfg.crt_epochs:
        classifier_retrain(model, loaders, ctx, cfg, seed)
    torch.save(model.state_dict(), out_dir / "best.pt")

    rv, rt = predict(model, loaders["val"], ctx, cfg), predict(model, loaders["test"], ctx, cfg)
    vm, tm = compute_metrics(rv), compute_metrics(rt)
    tm["minutes"] = round((time.time() - t0) / 60, 2)
    labels = np.arange(N_CLASSES)
    np.save(out_dir / "val_confusion.npy", confusion_matrix(rv["y"], rv["pred"], labels=labels))
    np.save(out_dir / "test_confusion.npy", confusion_matrix(rt["y"], rt["pred"], labels=labels))
    top3 = np.argsort(-rt["joint"], axis=1)[:, :3]
    pd.DataFrame({"y": rt["y"], "pred": rt["pred"], "top2": top3[:, 1], "top3": top3[:, 2]}).to_csv(out_dir / "test_preds.csv", index=False)
    curves.to_csv(out_dir / "curves.csv", index=False)
    (out_dir / "config.json").write_text(json.dumps(vars(cfg), indent=2, default=str))
    (out_dir / "val_metrics.json").write_text(json.dumps(vm, indent=2))
    (out_dir / "metrics.json").write_text(json.dumps(tm, indent=2))
    print(f"  → test top1 {tm['top1']:.3f} | macro-F1 {tm['macro_f1']:.3f} | tail recall {tm['recall_tail']:.3f} | {tm['minutes']}분")
    return tm

### 6-1. 불균형 해소 실험

`flat_ce`가 기준 모델(H0)이다. 나머지는 모두 Flat 구조에서 **불균형 기법 하나만** 바꾼다.

- 사후 보정(`posthoc_la`)과 cRT(`crt`)는 `flat_ce`의 체크포인트를 재사용하므로 따로 학습하지 않는다.
- 한 실험이 끝나야 다음 실험이 시작된다. 시간이 부족하면 `RUN_IMBALANCE` 목록에서 줄을 지우거나 `SEEDS`를 줄인다.

In [ ]:
DRW_AT = BASE["epochs"] * 2 // 3          # DRW: 전체 epoch의 2/3 지점부터 가중치를 켠다
IMBALANCE_EXPS = {
    "flat_ce":          dict(),                                                # 기준 (H0)
    "inv_a0.5":         dict(weighting="inv", alpha=0.5),
    "inv_a1":           dict(weighting="inv", alpha=1.0),                     # 수업 코드와 같은 방식
    "cb_b0.99":         dict(weighting="cb", beta=0.99),
    "cb_b0.999":        dict(weighting="cb", beta=0.999),
    "focal_g2":         dict(loss="focal", gamma=2.0),
    "cb_focal":         dict(weighting="cb", beta=0.999, loss="focal", gamma=2.0),
    "balanced_softmax": dict(loss="la", tau=1.0),
    "sampler_sqrt":     dict(q=0.5),
    "sampler_class":    dict(q=0.0),
    "drw_cb":           dict(weighting="cb", beta=0.999, drw_epoch=DRW_AT),
    "posthoc_la":       dict(from_run="flat_ce", posthoc_tau=1.0),            # 재학습 없음
    "crt":              dict(from_run="flat_ce", crt_epochs=10),              # 분류층만 재학습
}
RUN_IMBALANCE = list(IMBALANCE_EXPS)      # 일부만 돌리려면 여기서 이름을 고른다

for seed in SEEDS:
    for name in RUN_IMBALANCE:
        run(name, IMBALANCE_EXPS[name], seed)

### 6-2. 혼동 기반 그룹 만들기 (H2-b)

기준 모델 `flat_ce`의 **val 혼동 행렬**로 그룹을 만든다. test는 쓰지 않는다.
1. 행 정규화한 혼동 행렬 $C$를 대칭으로 만든다: $S = (C + C^\top)/2$. 서로 자주 헷갈린 차종 쌍일수록 값이 크다.
2. $S$를 "차종 간 유사도"로 보고 spectral clustering으로 K개 그룹을 만든다.
   - K 기본값은 6으로, 차체 유형 그룹 수와 같다. 그래서 "그룹을 어떻게 나누느냐"만 다른 공정한 비교가 된다.
3. 결과를 `RUNS_DIR/confusion_groups.json`에 저장한다. 계층 실험 `hier_conf`가 이 파일을 읽는다.

In [ ]:
def build_confusion_groups(base="flat_ce", seed=0, k=BASE["n_conf_groups"]):
    C = np.load(RUNS_DIR / base / f"seed{seed}" / "val_confusion.npy").astype(float)
    C = C / C.sum(1, keepdims=True).clip(min=1)
    S = (C + C.T) / 2
    np.fill_diagonal(S, 0)
    S = S + 1e-6                                        # 연결이 끊긴 차종이 없도록 아주 작은 값
    labels = SpectralClustering(n_clusters=k, affinity="precomputed", random_state=0).fit_predict(S)
    groups = [np.where(labels == g)[0].tolist() for g in range(k)]
    groups = [g for g in groups if g]
    (RUNS_DIR / "confusion_groups.json").write_text(json.dumps(
        {"source": f"{base}/seed{seed}/val_confusion.npy", "groups": groups,
         "names": [[CLASS_NAMES[c] for c in g] for g in groups]}, indent=2))
    return groups

conf_groups = build_confusion_groups()
for i, g in enumerate(conf_groups):
    print(f"그룹 {i} ({len(g)}개):", ", ".join(CLASS_NAMES[c] for c in g))

### 6-3. 계층형 분류 실험 (H1 · H2)

Flat 기준(`flat_ce`)과 비교한다. 네 모델은 **1단계 그룹을 무엇으로 정하느냐만** 다르다.

- 6-1에서 가장 좋았던 불균형 기법을 계층 모델에도 얹고 싶으면 `HIER_EXPS`에 한 줄을 추가한다. 예: `"hier_body_cb": dict(hier="body", weighting="cb", beta=0.999)`

In [ ]:
HIER_EXPS = {
    "hier_make": dict(hier="make"),          # H1: 제조사 10개 → 차종
    "hier_body": dict(hier="body"),          # H2-a: 차체 유형 6개 → 차종
    "hier_conf": dict(hier="confusion"),     # H2-b: 혼동 기반 그룹 → 차종
}
RUN_HIER = list(HIER_EXPS)

for seed in SEEDS:
    for name in RUN_HIER:
        run(name, HIER_EXPS[name], seed)

## 7. 결과 비교

`RUNS_DIR` 아래의 모든 `metrics.json`(test)을 모아 seed 평균 ± 표준편차를 낸다.
- **판단 기준:** 기준(`flat_ce`)보다 Macro-F1이 **seed 간 표준편차보다 크게, 일관되게** 오른 기법만 "개선"으로 본다.

In [ ]:
SCALARS = ["top1", "top3", "macro_f1", "bal_acc", "recall_head", "recall_medium", "recall_tail", "make_acc", "body_acc",
           "stage1_acc", "stage2_acc_given1", "err_from_stage1", "soft_top1", "minutes"]

def load_results():
    rows = []
    for f in sorted(RUNS_DIR.glob("*/seed*/metrics.json")):
        m = json.loads(f.read_text())
        row = {"exp": f.parent.parent.name, "seed": f.parent.name, **{k: m.get(k, np.nan) for k in SCALARS}}
        row["twin_confusion"] = np.nanmean(list(m["twin_confusion"].values()))
        rows.append(row)
    return pd.DataFrame(rows)

res = load_results()
order = [e for e in list(IMBALANCE_EXPS) + list(HIER_EXPS) if e in set(res.exp)]
summary = res.groupby("exp")[SCALARS[:-1] + ["twin_confusion"]].agg(["mean", "std"]).reindex(order)
summary.to_csv(RUNS_DIR / "summary.csv")
show = res.groupby("exp")[["top1", "macro_f1", "bal_acc", "recall_head", "recall_tail", "twin_confusion"]].mean().reindex(order)
show.style.format("{:.3f}").background_gradient(cmap="Blues", axis=0)

In [ ]:
agg = res.groupby("exp")[["top1", "macro_f1"]].agg(["mean", "std"]).reindex(order)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
x = np.arange(len(order))
for i, (col, c) in enumerate([("top1", "#8FB3E0"), ("macro_f1", "#2F6DB5")]):
    axes[0].bar(x + (i - 0.5) * 0.4, agg[(col, "mean")], 0.4, yerr=agg[(col, "std")].fillna(0), color=c, label=col, capsize=2)
axes[0].set_xticks(x, order, rotation=60, ha="right"); axes[0].legend()
axes[0].set(title="실험별 Top-1 vs Macro-F1 (seed 평균 ± 표준편차)", ylim=(0, 1))

rec = res.groupby("exp")[["recall_head", "recall_medium", "recall_tail"]].mean().reindex(order)
rec.columns = ["head", "medium", "tail"]
rec.plot(kind="bar", ax=axes[1], color=["#2F6DB5", "#8FB3E0", "#D9822B"], width=0.8)
axes[1].set(title="이미지 수 구간별 평균 recall", ylim=(0, 1), xlabel=""); axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

In [ ]:
def per_class_recall(exp):
    ms = [json.loads(f.read_text())["per_class_recall"] for f in (RUNS_DIR / exp).glob("seed*/metrics.json")]
    return np.nanmean(np.array(ms, dtype=float), axis=0)

if "flat_ce" in order:
    best = res[res.exp.isin(list(IMBALANCE_EXPS))].groupby("exp").macro_f1.mean().drop("flat_ce", errors="ignore")
    best = best.idxmax() if len(best) else "flat_ce"
    fig, ax = plt.subplots(figsize=(8, 5))
    for exp, c in [("flat_ce", "#999999"), (best, "#D9822B")]:
        ax.scatter(TRAIN_COUNTS, per_class_recall(exp), s=25, color=c, label=exp)
    ax.set(xscale="log", xlabel="train 이미지 수 (log)", ylabel="test recall",
           title=f"차종 크기 vs recall — 기준(flat_ce) vs 불균형 최선({best})")
    ax.legend(); plt.show()

In [ ]:
def plot_confusion(exp, ax):
    C = np.mean([np.load(f) for f in (RUNS_DIR / exp).glob("seed*/test_confusion.npy")], axis=0)
    C = C / C.sum(1, keepdims=True).clip(min=1)
    idx = CLASS_INFO.sort_values(["body_idx", "make_idx"]).model_idx.to_numpy()   # 차체 유형 순 정렬
    sns.heatmap(C[np.ix_(idx, idx)], ax=ax, cmap="Blues", vmin=0, vmax=0.3, cbar=False,
                xticklabels=[CLASS_NAMES[i] for i in idx], yticklabels=[CLASS_NAMES[i] for i in idx])
    ax.set(title=f"{exp} (행 정규화, 차체 유형 순)", xlabel="예측", ylabel="정답")
    ax.tick_params(labelsize=5)

shown = [e for e in ["flat_ce", "hier_body"] if e in order]
if shown:
    fig, axes = plt.subplots(1, len(shown), figsize=(9 * len(shown), 8), squeeze=False)
    for ax, e in zip(axes[0], shown):
        plot_confusion(e, ax)
    plt.tight_layout(); plt.show()

In [ ]:
hier_rows = [e for e in ["flat_ce", "hier_make", "hier_body", "hier_conf"] if e in order]
if hier_rows:
    display(res.groupby("exp")[["top1", "macro_f1", "make_acc", "body_acc", "stage1_acc", "stage2_acc_given1",
                                "err_from_stage1", "soft_top1", "twin_confusion"]].mean().reindex(hier_rows).round(3))

**읽는 법 (계층 표)**
- `stage1_acc`: 1단계 그룹 정확도
- `stage2_acc_given1`: 1단계를 맞혔을 때 2단계 정확도
- `err_from_stage1`: 전체 오답 중 1단계에서 이미 틀린 비율. 높을수록 오류 전파가 문제라는 뜻이다.
- `soft_top1`과 `top1`의 차이: hard routing 대신 soft 결합을 썼다면 얼마나 달라졌을지
- `twin_confusion`: 형제차·트림 쌍끼리 서로 착각한 비율. H2의 예측대로라면 `hier_make`에서 가장 높아야 한다.